In [9]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [12]:
# Load IMDB text files manually

def load_imdb_split(folder):
    texts = []
    labels = []

    for label_type in ["pos", "neg"]:
        path = os.path.join(folder, label_type)

        for fname in os.listdir(path):
            if fname.endswith(".txt"):
                with open(os.path.join(path, fname), encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(1 if label_type == "pos" else 0)

    return texts, np.array(labels, dtype=np.float32)


base_dir = "./aclImdb"  # <-- change if needed

train_texts, train_labels = load_imdb_split(os.path.join(base_dir, "train"))
test_texts, test_labels = load_imdb_split(os.path.join(base_dir, "test"))

In [14]:
# Bag-of-words vectorization (10k words)

max_words = 10000

vectorizer = CountVectorizer(
    max_features=max_words,
    binary=True,
    stop_words="english"
)

x_train = vectorizer.fit_transform(train_texts).toarray().astype(np.float32)
x_test = vectorizer.transform(test_texts).toarray().astype(np.float32)

In [15]:
# Validation split

x_train, x_val, y_train, y_val = train_test_split(
    x_train, train_labels, test_size=10000, random_state=42, stratify=train_labels
)

In [16]:
# Convert to tensors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_train = torch.tensor(x_train).to(device)
y_train = torch.tensor(y_train).unsqueeze(1).to(device)

x_val = torch.tensor(x_val).to(device)
y_val = torch.tensor(y_val).unsqueeze(1).to(device)

In [17]:
hidden_nodes = 64
 
class IMDBNet(nn.Module):
    def __init__(self, dropout=0.0):
        super(IMDBNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(max_words, hidden_nodes),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_nodes, hidden_nodes),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_nodes, 1),
            nn.Sigmoid()
        )
 
    def forward(self, x):
        return self.net(x)
 
model = IMDBNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [18]:
def train_model(
    x_train, y_train, x_val, y_val,
    l1_lambda=0.0,
    l2_lambda=0.0,
    dropout=0.0,
    epochs=20,
    batch_size=512
):
    device = x_train.device
    model = IMDBNet(dropout=dropout).to(device)
 
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=l2_lambda)
 
    train_losses = []
    val_losses = []
 
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(x_train.size(0), device=device)
        epoch_loss = 0.0
        n_batches = 0
 
        for i in range(0, x_train.size(0), batch_size):
            idx = perm[i:i + batch_size]
            xb = x_train[idx]
            yb = y_train[idx]
 
            preds = model(xb)
            loss = criterion(preds, yb)
 
            # L1 regularization (added to loss)
            if l1_lambda > 0:
                l1_penalty = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_penalty
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            epoch_loss += loss.item()
            n_batches += 1
 
        train_loss = epoch_loss / max(n_batches, 1)
        train_losses.append(train_loss)
 
        model.eval()
        with torch.no_grad():
            val_preds = model(x_val)
            val_loss = criterion(val_preds, y_val).item()
            val_losses.append(val_loss)
 
        print(f"Epoch {epoch+1:02d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")
 
    history = {
        "train_loss": train_losses,
        "val_loss": val_losses
    }
 
    metrics = {
        "best_val_loss": min(val_losses) if val_losses else None,
        "last_val_loss": val_losses[-1] if val_losses else None
    }
 
    return model, history, metrics

In [19]:
trained_models = {}
results = {}
 
trained_models["baseline"], _, results["baseline"] = train_model(x_train, y_train, x_val, y_val)
 
trained_models["l2"], _, results["l2"] = train_model(
x_train, y_train, x_val, y_val,
l2_lambda=1e-3
)
 
trained_models["l1"], _, results["l1"] = train_model(
x_train, y_train, x_val, y_val,
l1_lambda=1e-5
)
 
trained_models["dropout"], _, results["dropout"] = train_model(
x_train, y_train, x_val, y_val,
dropout=0.5)

Epoch 01 | Train loss: 0.5534 | Val loss: 0.3648
Epoch 02 | Train loss: 0.2476 | Val loss: 0.2979
Epoch 03 | Train loss: 0.1464 | Val loss: 0.3374
Epoch 04 | Train loss: 0.0946 | Val loss: 0.3876
Epoch 05 | Train loss: 0.0605 | Val loss: 0.4707
Epoch 06 | Train loss: 0.0377 | Val loss: 0.5925
Epoch 07 | Train loss: 0.0226 | Val loss: 0.6762
Epoch 08 | Train loss: 0.0148 | Val loss: 0.7733
Epoch 09 | Train loss: 0.0091 | Val loss: 0.8518
Epoch 10 | Train loss: 0.0054 | Val loss: 0.8871
Epoch 11 | Train loss: 0.0034 | Val loss: 0.9637
Epoch 12 | Train loss: 0.0024 | Val loss: 1.0598
Epoch 13 | Train loss: 0.0018 | Val loss: 1.1028
Epoch 14 | Train loss: 0.0014 | Val loss: 1.1585
Epoch 15 | Train loss: 0.0012 | Val loss: 1.2121
Epoch 16 | Train loss: 0.0010 | Val loss: 1.2567
Epoch 17 | Train loss: 0.0008 | Val loss: 1.3068
Epoch 18 | Train loss: 0.0007 | Val loss: 1.3392
Epoch 19 | Train loss: 0.0006 | Val loss: 1.3706
Epoch 20 | Train loss: 0.0005 | Val loss: 1.3917
Epoch 01 | Train los

In [22]:
# Training loop
epochs = 20
batch_size = 512
train_losses = []
val_losses = []
 
for epoch in range(epochs):
    perm = torch.randperm(x_train.size(0))
    epoch_loss = 0
    for i in range(0, x_train.size(0), batch_size):
        idx = perm[i:i + batch_size]
 
        xb = x_train[idx]
        yb = y_train[idx]
 
        preds = model(xb)
        loss = criterion(preds, yb)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        epoch_loss += loss.item()
 
train_losses.append(epoch_loss)
 
with torch.no_grad():
    val_preds = model(x_val)
    val_loss = criterion(val_preds, y_val)
    val_losses.append(val_loss.item())
 
print(f"Epoch {epoch+1:02d} | Train loss: {epoch_loss :.3f} | Val loss: {val_loss :.3f}")

Epoch 20 | Train loss: 0.001 | Val loss: 2.229
